# Results: tables and target vs. non-target figure
Reads the metric pickles written by `evaluate_metrics.ipynb` and produces
1. a table with the conventional metrics next to their list-aware counterparts, and
2. a scatter of each conventional metric on the target vs. the same metric on the non-target items (points near or above $y=x$ mean the explanation moves the rest of the list as much as the target).

Outputs go to `results/<dataset>/`.

In [ ]:
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import *

In [ ]:
data_name = "ML1M"          # ML1M, Yahoo, Pinterest
recommender_name = "MLP"    # MLP (= MF in the paper), VAE
k_e = PI                    # explanation size to report (1..PI)

DISPLAY_NAMES = {'jaccard': 'Jaccard', 'cosine': 'Cosine', 'lime': 'LIME', 'shap': 'SHAP',
                 'accent': 'ACCENT', 'spinrec': 'SPINRec', 'lxr': 'LXR'}

out_dir = RESULTS_DIR / data_name
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
all_metrics = {}
for name in EXPLAINERS:
    path = metrics_path(data_name, recommender_name, name)
    if path.exists():
        with open(path, 'rb') as f:
            all_metrics[name] = pickle.load(f)
    else:
        print(f'missing: {path}')
print('loaded:', list(all_metrics))

## Table: conventional vs. list-aware metrics at $K_e$

In [ ]:
TABLE_COLUMNS = {
    # conventional                      # list-aware counterpart
    'POS_at_20': 'POS@20 ↑',            'COV_CV': 'Validity ↑',
    'DEL': 'DEL ↓',                     'COV_NEC': 'Nec. cov ↑',  'DELTA_NEC': 'Nec. Δ ↑',
    'INS': 'INS ↑',                     'COV_SUF': 'Suf. cov ↑',  'DELTA_SUF': 'Suf. Δ ↑',
    'CDCG': 'CDCG ↓',                   'COV_RD': 'RD cov ↑',     'DELTA_RD': 'RD Δ ↑',
}

rows = {}
for name, m in all_metrics.items():
    rows[DISPLAY_NAMES.get(name, name)] = {label: float(np.asarray(m[key])[k_e - 1])
                                           for key, label in TABLE_COLUMNS.items() if key in m}
table = pd.DataFrame(rows).T
table.to_csv(out_dir / f'{recommender_name}_table_Ke{k_e}.csv')
table.round(3)

## Figure: target vs. non-target effect

In [ ]:
PANELS = [
    ('POS_at_20', 'NONTARGET_POS_at_20', 'POS@20 (target)', 'POS@20 (non-target)'),
    ('CDCG', 'NONTARGET_CDCG', 'CDCG (target)', 'CDCG (non-target)'),
]
markers = ['o', 's', '^', 'D', 'P', 'X', 'v']

fig, axes = plt.subplots(1, len(PANELS), figsize=(5 * len(PANELS), 4.2))
for ax, (t_key, nt_key, xlabel, ylabel) in zip(axes, PANELS):
    for i, (name, m) in enumerate(all_metrics.items()):
        x, y = np.asarray(m[t_key])[k_e - 1], np.asarray(m[nt_key])[k_e - 1]
        ax.scatter(x, y, marker=markers[i % len(markers)], s=60, label=DISPLAY_NAMES.get(name, name), zorder=3)
    ax.plot([0, 1], [0, 1], color='grey', linestyle='--', linewidth=1, zorder=1)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8, loc='best')
fig.suptitle(f'{data_name}, {recommender_name}, $K_e$={k_e}')
fig.tight_layout()
fig.savefig(out_dir / f'{recommender_name}_target_vs_nontarget_Ke{k_e}.pdf', bbox_inches='tight')
plt.show()